In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
from scipy.signal import butter, sosfilt
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, f1_score

CFG = dict(
    train_csv="metadata.csv",
    test_csv="sample_submission.csv",
    train_dir=r"C:\Users\local_ypfz2f8\OneDrive\Dokumen\UTScode\data\schematics-2026\train\audio",
    test_dir=r"C:\Users\local_ypfz2f8\OneDrive\Dokumen\UTScode\data\schematics-2026\test\audio",
    ext=".flac",
    sr=22050,
    max_sec=10,
    classes=["Bondol Jawa", "Cendrawasih", "Kutilang"],
    n_folds=5,
    unknown_share=0.30,
    seed=42,
)

BEST_PARAMS = {
    "n_estimators": 500,
    "max_depth": 15,
    "min_samples_leaf": 10,
    "max_features": 0.3,
    "class_weight": "balanced"
}

def find_paths(df, root, ext):
    idx = {}
    for dp, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(ext):
                idx[os.path.splitext(f)[0]] = os.path.join(dp, f)
    ids = [str(f) for f in df["file_id"]]
    ids_clean = [i[:-len(ext)] if i.lower().endswith(ext) else i for i in ids]
    return [idx[i] for i in ids_clean if i in idx]

def get_feats(audio_chunk, sr):
    if len(audio_chunk) < sr:
        audio_chunk = np.pad(audio_chunk, (0, sr - len(audio_chunk)))
        
    S = librosa.feature.melspectrogram(y=audio_chunk, sr=sr, n_mels=64, fmin=500, fmax=10000)
    
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(S + 1e-6), n_mfcc=20)
    if mfcc.shape[1] < 9:
        mfcc = np.pad(mfcc, ((0, 0), (0, 9 - mfcc.shape[1])))
        
    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2) 
    
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr)
    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(audio_chunk)
    
    feats = []
    for f in [mfcc, mfcc_delta, mfcc_delta2, centroid, bandwidth, rolloff, contrast, zcr]:
        if f.size == 0: continue
        feats.extend([np.mean(f, axis=1), np.std(f, axis=1), np.max(f, axis=1)])
        
    try:
        S_pcen = librosa.pcen(S, sr=sr)
        feats.extend([np.mean(S_pcen, axis=1), np.max(S_pcen, axis=1)])
    except:
        pass
        
    return np.concatenate(feats)

def extract(path):
    sr = CFG["sr"]
    try:
        y, _ = librosa.load(path, sr=sr, mono=True, duration=CFG["max_sec"])
    except: 
        return None

    sos = butter(4, [500, 10000], btype='band', fs=sr, output='sos')
    y = sosfilt(sos, y).astype(np.float32)
    
    if len(y) < sr * 2: 
        y = np.pad(y, (0, sr * 2 - len(y)))
    
    intervals = librosa.effects.split(y, top_db=20) 
    
    if len(intervals) > 0:
        y_active = np.concatenate([y[s:e] for s, e in intervals])
    else:
        y_active = y
        
    if len(y_active) < sr: 
        y_active = np.pad(y_active, (0, sr - len(y_active)))
        
    global_feats = get_feats(y_active, sr)
    
    chunk_size = sr * 2 
    if len(y_active) > chunk_size:
        rms = librosa.feature.rms(y=y_active, frame_length=chunk_size, hop_length=chunk_size // 2)[0]
        best_idx = np.argmax(rms)
        start = best_idx * (chunk_size // 2)
        end = min(start + chunk_size, len(y_active))
        y_focus = y_active[start : end]
        focus_feats = get_feats(y_focus, sr)
    else:
        focus_feats = global_feats

    return np.concatenate([global_feats, focus_feats]).astype(np.float32)

def get_features(paths, name):
    cache_file = f"{name}_v5_upgrade.npy"
    if os.path.exists(cache_file):
        X = np.load(cache_file)
        if X.shape[0] == len(paths): 
            print(f"Cache {cache_file} dimuat.")
            return X
            
    print(f"Ekstraksi fitur untuk {name}...")
    results = Parallel(n_jobs=-1)(delayed(extract)(p) for p in paths)
    X = np.stack([r for r in results if r is not None])
    np.save(cache_file, X)
    return X

def main():
    train = pd.read_csv(CFG["train_csv"])
    test = pd.read_csv(CFG["test_csv"])
    cls2id = {c: i for i, c in enumerate(CFG["classes"])}
    y = train["label"].map(cls2id).values
    groups = train["source_group"].values

    X = get_features(find_paths(train, CFG["train_dir"], CFG["ext"]), "feat_train")
    Xte = get_features(find_paths(test, CFG["test_dir"], CFG["ext"]), "feat_test")
    print("Bentuk fitur:", X.shape, Xte.shape)

    X = np.nan_to_num(X, nan=0.0)
    Xte = np.nan_to_num(Xte, nan=0.0)

    print("\nTraining model ExtraTrees (Konfigurasi 69%)...")
    model = ExtraTreesClassifier(
        n_estimators=BEST_PARAMS["n_estimators"],
        max_depth=BEST_PARAMS["max_depth"],
        min_samples_leaf=BEST_PARAMS["min_samples_leaf"],
        max_features=BEST_PARAMS["max_features"],
        class_weight=BEST_PARAMS["class_weight"],
        random_state=CFG["seed"],
        n_jobs=-1
    )
    
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG["seed"])
    pred_cv = cross_val_predict(model, X, y, groups=groups, cv=cv, n_jobs=1)
    print("\nLaporan CV:")
    print(classification_report(y, pred_cv, target_names=CFG["classes"], zero_division=0))
    print(f"CV Macro-F1: {f1_score(y, pred_cv, average='macro', zero_division=0):.4f}")

    print("\nTraining Final Model...")
    model.fit(X, y)
    p = model.predict_proba(Xte)
    msp = p.max(1)

    thr = float(np.quantile(msp, CFG["unknown_share"]))
    pred = np.where(msp < thr, "Unknown", np.array(CFG["classes"])[p.argmax(1)])
    
    print(f"\nThreshold Unknown (Share {CFG['unknown_share']*100:.0f}%): {thr:.3f}")
    print("Distribusi prediksi test:")
    print(pd.Series(pred).value_counts())
    
    sub = pd.DataFrame({"file_id": test["file_id"], "label": pred})
    sub.to_csv("submission_v5_69percent.csv", index=False)
    print("\nSaved: submission_v5_69percent.csv")

if __name__ == "__main__":
    main()

Cache feat_train_v5_upgrade.npy dimuat.
Cache feat_test_v5_upgrade.npy dimuat.
Bentuk fitur: (1826, 682) (814, 682)

Training model ExtraTrees (Konfigurasi 69%)...

Laporan CV:
              precision    recall  f1-score   support

 Bondol Jawa       0.98      0.96      0.97      1124
 Cendrawasih       0.75      0.82      0.78        83
    Kutilang       0.94      0.97      0.95       619

    accuracy                           0.96      1826
   macro avg       0.89      0.92      0.90      1826
weighted avg       0.96      0.96      0.96      1826

CV Macro-F1: 0.9020

Training Final Model...

Threshold Unknown (Share 30%): 0.624
Distribusi prediksi test:
Unknown        244
Kutilang       240
Bondol Jawa    215
Cendrawasih    115
Name: count, dtype: int64

Saved: submission_v5_69percent.csv
